# RAG 코드블록 단위 빈칸채우기 문제 및 해설

AI 인증시험 RAG 출제포인트와 제공된 RAG 실습 노트북을 기준으로, 관련 있는 코드블록만 선별해 구성했습니다.

## 선별 기준

- RAG 구성 요소와 흐름: document loading, chunking, indexing, retrieval, response generation.
- 관련 라이브러리 사용: LlamaIndex, embedding retriever, query engine, MCP tool/agent.
- 평가와 고도화: CRAG evaluation, KG/API 기반 retrieval, MCP 기반 tool calling.
- API key 설정, 단순 설치, 단순 출력 확인처럼 시험 포인트와 직접 관련이 낮은 코드는 제외했습니다.

## LlamaIndex 기반 RAG

### 문제 1. Document Loading / Indexing / Query Engine

LlamaIndex에서 문서를 읽고 vector index를 만든 뒤 query engine으로 질의하는 기본 RAG 흐름입니다. RAG 구성 요소의 순서에 맞게 빈칸을 완성하세요.

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = ____

index = ____

query_engine = ____

response = ____
print(response)

**정답**

1. `SimpleDirectoryReader("data").load_data()`

   해설: data 폴더의 파일을 LlamaIndex Document 객체 목록으로 읽어옵니다.

2. `VectorStoreIndex.from_documents(documents)`

   해설: 문서를 embedding 기반 vector index로 변환해 검색 가능한 구조를 만듭니다.

3. `index.as_query_engine()`

   해설: index를 검색과 응답 생성을 함께 수행하는 query engine으로 바꿉니다.

4. `query_engine.query("What is the first programs the author tried writing?")`

   해설: 질문을 query engine에 전달하면 관련 문서를 검색하고 답변을 생성합니다.

**전체 해설**

RAG의 기본 흐름은 문서 로딩, 인덱싱, 검색, 응답 생성입니다. SimpleDirectoryReader가 데이터를 Document로 만들고, VectorStoreIndex가 검색 가능한 벡터 인덱스를 구성하며, query_engine이 retrieval과 generation을 연결합니다.

### 문제 2. Chunking With SentenceSplitter

문서를 chunk로 나누고 chunk 크기와 overlap을 조절해 index를 만드는 코드입니다. chunking과 transformation 적용 부분을 완성하세요.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex

parser = ____
nodes = ____

text_splitter = ____
index = ____

**정답**

1. `SentenceSplitter(chunk_size=1024, chunk_overlap=200)`

   해설: 큰 chunk와 overlap을 사용해 문맥을 비교적 넓게 유지하며 문서를 나눕니다.

2. `parser.get_nodes_from_documents(documents)`

   해설: Document 객체들을 실제 검색 단위인 node/chunk 목록으로 변환합니다.

3. `SentenceSplitter(chunk_size=200, chunk_overlap=50)`

   해설: 더 짧은 chunk 설정을 만들어 retrieval 단위를 세밀하게 조절합니다.

4. `VectorStoreIndex.from_documents(documents=documents, transformations=[text_splitter])`

   해설: 인덱스 생성 시 transformation으로 splitter를 넘겨 chunking이 적용된 index를 만듭니다.

**전체 해설**

RAG에서 chunk_size와 chunk_overlap은 검색 품질에 직접 영향을 줍니다. chunk가 너무 크면 불필요한 문맥이 섞이고, 너무 작으면 답변에 필요한 정보가 끊길 수 있습니다.

### 문제 3. Retriever And Context Prompt

query engine이 자동으로 처리하던 retrieval과 generation을 분리해 보는 코드입니다. 검색 결과를 context로 묶어 LLM에 전달하는 부분을 완성하세요.

In [ ]:
retriever = ____

ret_passages = ____

ret_context = ""
for ret_result in ret_passages:
  ret_context += ____

question = f"""Context information is below.
---------------------
{ret_context}
---------------------
Given the context information and not prior knowledge,
answer the query and the reasons of answer.
Query: Who is the author?
Answer:
"""

print(____)

**정답**

1. `index.as_retriever()`

   해설: index에서 검색 기능만 분리한 retriever 객체를 만듭니다.

2. `retriever.retrieve("Who is the author?")`

   해설: 질문과 유사한 node/chunk들을 검색합니다.

3. `ret_result.text`

   해설: 검색된 passage의 실제 텍스트를 context 문자열에 누적합니다.

4. `generate_answer(question)`

   해설: 검색 context가 포함된 prompt를 LLM에 전달해 최종 답변을 생성합니다.

**전체 해설**

RAG는 retrieve와 generate가 결합된 구조입니다. 검색된 passage를 prompt에 넣으면 LLM이 사전지식이 아니라 제공된 context를 근거로 답하도록 유도할 수 있습니다.

### 문제 4. Custom Query Engine

LlamaIndex의 CustomQueryEngine으로 retriever와 synthesizer를 직접 연결하는 코드입니다. custom_query에서 검색과 합성을 수행하는 부분을 완성하세요.

In [ ]:
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.response_synthesizers import BaseSynthesizer
from llama_index.core import get_response_synthesizer

class StandardQueryEngine(CustomQueryEngine):
    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer

    def custom_query(self, query_str: str):
        nodes = ____
        response_obj = ____
        return response_obj

retriever = ____
synthesizer = ____
query_engine = StandardQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
)

**정답**

1. `self.retriever.retrieve(query_str)`

   해설: 입력 질문에 대해 관련 node를 검색합니다.

2. `self.response_synthesizer.synthesize(query_str, nodes)`

   해설: 검색된 node들을 질문과 함께 합성해 최종 응답 객체를 만듭니다.

3. `index.as_retriever()`

   해설: vector index에서 retriever를 가져옵니다.

4. `get_response_synthesizer(response_mode="compact")`

   해설: 검색 결과를 compact 방식으로 합성하는 response synthesizer를 생성합니다.

**전체 해설**

CustomQueryEngine은 RAG 내부의 검색기와 응답 합성기를 명시적으로 조립하게 해줍니다. 시험 관점에서는 RAG 구성 요소가 어떻게 연결되는지 이해하는 데 중요합니다.

## From-Scratch RAG

### 문제 5. RAG Class Structure

검색 단계와 생성 단계를 직접 구현한 RAG 클래스입니다. retrieve, generate_response, query의 연결 흐름을 완성하세요.

In [ ]:
class RAG_from_scratch:
    def retrieve(self, query: str) -> list:
        results = ____
        return results

    def generate_response(self, query: str, context_str: list) -> str:
        completion = oai_client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0,
            messages=[
                {
                    "role": "user",
                    "content": f"We have provided context information below.\n"
                               f"---------------------\n"
                               f"{context_str}"
                               f"\n---------------------\n"
                               f"Given this information, please answer the question: {query}",
                }
            ],
        ).choices[0].message.content
        return completion

    def query(self, query: str) -> str:
        context_str = ____
        completion = ____
        return completion

**정답**

1. `query_engine.query(query)`

   해설: 질문을 기존 query_engine에 넣어 관련 context를 검색 또는 생성 결과 형태로 가져옵니다.

2. `self.retrieve(query)`

   해설: query 메서드에서 먼저 검색 단계를 호출해 context를 얻습니다.

3. `self.generate_response(query, context_str)`

   해설: 검색된 context와 원 질문을 generation 단계에 넘겨 최종 답변을 만듭니다.

**전체 해설**

RAG를 직접 구현할 때도 핵심은 retrieve와 generate의 분리입니다. query는 두 단계를 순서대로 연결하는 orchestration 역할을 합니다.

## CRAG 데이터 전처리와 Retrieval

### 문제 6. HTML Parsing And Chunk Extraction

CRAG search_results의 HTML에서 텍스트를 추출하고 sentence 단위 chunk로 나누는 전처리 코드입니다. RAG 입력 데이터 구성 흐름에 맞게 빈칸을 완성하세요.

In [ ]:
from bs4 import BeautifulSoup
from blingfire import text_to_sentences_and_offsets

def parse_htmls(search_results):
    all_documents = []
    for html_text in search_results:
        soup = ____
        text = ____
        all_documents.append(text)
    return all_documents

def extract_chunks(all_documents):
    all_chunks = []
    for document in all_documents:
        if not document:
            all_chunks.append("")
        else:
            _, offsets = ____
            for start, end in offsets:
                chunk = ____
                all_chunks.append(chunk)
    return all_chunks

**정답**

1. `BeautifulSoup(html_text["page_result"], features="lxml")`

   해설: search result 안의 HTML 문자열을 BeautifulSoup으로 파싱합니다.

2. `soup.get_text(" ", strip=True)`

   해설: HTML 태그를 제거하고 공백 기준의 순수 텍스트를 추출합니다.

3. `text_to_sentences_and_offsets(document)`

   해설: 문서 텍스트에서 문장 단위 offset을 추출합니다.

4. `document[start:end][:MAX_CONTEXT_SENTENCE_LENGTH]`

   해설: 각 문장 offset을 이용해 chunk를 만들고 최대 길이를 제한합니다.

**전체 해설**

RAG는 검색 가능한 텍스트 단위가 필요합니다. HTML 원문을 그대로 쓰지 않고 텍스트를 뽑은 뒤 문장 또는 chunk 단위로 나누어야 embedding/retrieval에 사용할 수 있습니다.

### 문제 7. Embedding Retriever From Scratch

OpenAI embedding과 cosine similarity로 top-k chunk를 검색하는 retriever입니다. embedding 생성, query embedding, similarity 계산, top-k 선택을 완성하세요.

In [ ]:
class BaseRetriever:
    def __init__(self):
        self.client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    def embed_text(self, texts):
        if isinstance(texts, str):
            texts = [texts]

        response = self.client.embeddings.create(
            model="text-embedding-3-small",
            input=texts,
        )
        embeddings = ____
        return np.array(embeddings)

    def retrieve(self, query, search_results, topk):
        all_documents = parse_htmls(search_results)
        all_chunks = extract_chunks(all_documents)

        all_embeddings = ____
        query_embedding = ____

        cosine_scores = ____
        top_k_indices = ____
        top_k_chunks = ____

        return top_k_chunks

**정답**

1. `[np.array(item.embedding) for item in response.data]`

   해설: embedding API 응답에서 각 입력 텍스트의 embedding 벡터를 꺼냅니다.

2. `self.embed_text(all_chunks)`

   해설: 검색 대상 chunk들을 embedding 벡터로 변환합니다.

3. `self.embed_text(query)[0]`

   해설: 질문도 같은 embedding 공간의 벡터로 변환합니다.

4. `np.dot(all_embeddings, query_embedding) / (np.linalg.norm(all_embeddings, axis=1) * np.linalg.norm(query_embedding))`

   해설: chunk embedding과 query embedding 사이의 cosine similarity를 계산합니다.

5. `(-cosine_scores).argsort()[:topk]`

   해설: 유사도가 높은 순서대로 정렬해 상위 top-k 인덱스를 선택합니다.

6. `np.array(all_chunks)[top_k_indices]`

   해설: 선택된 인덱스에 해당하는 chunk 텍스트를 반환합니다.

**전체 해설**

vector retrieval은 query와 문서 chunk를 같은 embedding 공간에 놓고 유사도를 계산하는 방식입니다. cosine similarity가 높을수록 질문과 의미적으로 가까운 chunk로 볼 수 있습니다.

### 문제 8. Reader Prompt And RAG Inference

검색된 chunk를 references로 prompt에 넣고 답변을 생성하는 Reader와, retriever/reader를 결합한 RAG 클래스입니다. 빈칸을 완성하세요.

In [ ]:
class Reader:
  def generate_response(self, query: str, top_k_chunks: list) -> str:
      llm_input = ____
      completion = oai_client.chat.completions.create(
          model="gpt-3.5-turbo",
          temperature=0,
          messages=llm_input,
      ).choices[0].message.content
      return completion

  def prompt_generator(self, query, top_k_chunks):
      references = ""
      if len(top_k_chunks) > 0:
          references += "# References \n"
          for chunk_id, chunk in enumerate(top_k_chunks):
              references += ____

      references = ____
      user_message = f"{references}\n------\n\n"
      user_message += "Using only the references listed above, answer the following question: \n"
      user_message += f"Question: {query}\n"

      llm_input = [
        {"role": "system", "content": self.system_prompt},
        {"role": "user", "content": user_message},
      ]
      return llm_input

class RAG:
    def __init__(self):
        self.retriever = LlamaIndexRetriever()
        self.reader = Reader()

    def inference(self, query, search_results, topk):
        retrieved_results = ____
        answer = ____
        return answer, retrieved_results

**정답**

1. `self.prompt_generator(query, top_k_chunks)`

   해설: 질문과 검색 결과를 LLM 메시지 형식으로 변환합니다.

2. `f"- {chunk.strip()}\n"`

   해설: 검색된 chunk를 reference 목록 형식으로 prompt에 추가합니다.

3. `references[:MAX_CONTEXT_REFERENCES_LENGTH]`

   해설: LLM 입력 길이를 넘지 않도록 references 길이를 제한합니다.

4. `self.retriever.retrieve(query, search_results, topk)`

   해설: 질문과 search_results에서 관련 chunk를 검색합니다.

5. `self.reader.generate_response(query, retrieved_results)`

   해설: 검색된 chunk를 근거로 최종 답변을 생성합니다.

**전체 해설**

Reader는 검색 결과를 LLM이 읽을 수 있는 prompt로 포장합니다. RAG 클래스는 retriever와 reader를 결합해 검색 결과에 기반한 답변 생성을 수행합니다.

## RAG 평가

### 문제 9. CRAG Evaluation

CRAG 데이터셋에서 모델 예측이 ground truth와 맞는지 LLM judge로 평가하고 점수를 계산하는 코드입니다. 평가 흐름에 필요한 빈칸을 완성하세요.

In [ ]:
def parse_response(response):
    try:
        response = response.lower()
        model_resp = ____
        answer = -1
        if "accuracy" in model_resp and (
            model_resp["accuracy"] is True
            or (
                isinstance(model_resp["accuracy"], str)
                and model_resp["accuracy"].lower() == "true"
            )
        ):
            answer = 1
        else:
            raise ValueError(f"Could not parse answer from response: {model_resp}")
        return answer
    except:
        return -1

def CRAG_evaluation(question, ground_truth, prediction):
  context_template = f"""Question: {question}
  List of Ground Truth answers: {ground_truth}
  Model Prediction: {prediction}
  """
  evaluation_result = ____
  eval_res = ____
  return eval_res

CRAG_score = ____

**정답**

1. `json.loads(response)`

   해설: LLM judge가 반환한 JSON 문자열을 파싱합니다.

2. `generate_answer(user_prompt=context_template, system_prompt=INSTRUCTIONS)`

   해설: 평가 지시문과 질문/정답/예측을 LLM judge에 전달합니다.

3. `parse_response(evaluation_result)`

   해설: judge 응답에서 accuracy 여부를 숫자 평가값으로 변환합니다.

4. `n_correct_exact + 0.5*n_correct - n_hallucinate`

   해설: 정확히 맞춘 답, 부분 인정 답, hallucination penalty를 반영해 CRAG 점수를 계산합니다.

**전체 해설**

RAG 평가에서는 단순 문자열 일치만으로 부족할 수 있어 LLM judge를 사용합니다. CRAG score는 정확도뿐 아니라 모르겠다고 답한 경우와 hallucination을 구분해 평가합니다.

## KG / MCP 결합 RAG

### 문제 10. KG Retrieval Combined With Search Results

Knowledge Graph 결과와 일반 search result retrieval을 결합하는 RAG 구조입니다. 금융 질의처럼 구조화 지식이 중요한 경우 KG 결과를 함께 넣는 흐름을 완성하세요.

In [ ]:
class RAGWithSRKG:
    def __init__(self):
        self.retriever = LlamaIndexRetriever()
        self.kg_query_engine = KGQueryEngine()
        self.reader = Reader()

    def inference(self, query, search_results, topk):
        retrieved_results = ____

        kg_results, is_finance = ____

        combined_results = ____
        combined_results.extend(____)

        answer = ____

        return answer, combined_results

**정답**

1. `self.retriever.retrieve(query, search_results, topk)`

   해설: 일반 search_results에서 query와 관련 있는 chunk를 검색합니다.

2. `self.kg_query_engine.query(query)`

   해설: query를 구조화 질의로 변환해 KG에서 관련 결과를 가져옵니다.

3. `[kg_results]`

   해설: KG 결과를 검색 context 목록의 첫 원소로 배치합니다.

4. `retrieved_results`

   해설: 일반 retrieval 결과를 KG 결과 뒤에 추가해 context를 확장합니다.

5. `self.reader.generate_response(query, combined_results)`

   해설: KG와 search retrieval 결과를 함께 근거로 최종 답변을 생성합니다.

**전체 해설**

RAG는 비정형 문서 검색뿐 아니라 KG/API 같은 구조화 소스와도 결합할 수 있습니다. 이 구조는 검색 결과와 KG 결과를 모두 references로 넣어 답변 정확도를 높이는 방식입니다.

### 문제 11. MCP Tool Agent And RAG With MCP

MCP tool을 LlamaIndex FunctionAgent에 연결하고, MCP 기반 KG 검색 결과를 Reader에 넘기는 구조입니다. tool 초기화, agent 실행, 최종 RAG inference 흐름을 완성하세요.

In [ ]:
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.workflow import Context

external_mcp_server = "http://server/sse"
mcp_client = ____
mcp_tool = ____

class KGQueryEngineWithMCP:
    def __init__(self, mcp_tool_spec: McpToolSpec, model: str, llm=None):
        self.llm = llm or OpenAI(model=model, temperature=0)
        self.mcp_tool_spec = mcp_tool_spec
        self.agent = None
        self.agent_context = None

    async def init_agent(self):
        tools = ____
        self.agent = FunctionAgent(
            name="Agent",
            description="An agent that can work with Our Knowledge Graph api.",
            tools=tools,
            llm=self.llm,
            system_prompt=SYSTEM_PROMPT,
        )
        self.agent_context = ____

    async def query(self, question: str, verbose: bool = False) -> str:
        handler = ____
        response = ____
        return str(response)

class RAGwithMCP:
    def __init__(self, mcp_tool):
        self.mcp_application = KGQueryEngineWithMCP(mcp_tool, model="gpt-4o-mini")
        self.reader = Reader()

    async def retrieve(self, query: str, query_time: str, search_results: list, topk: int):
        await self.mcp_application.init_agent()
        full_query = f"Query: {query}\nQuery time: {query_time}"
        mcp_result = ____
        return mcp_result

    async def inference(self, query: str, search_results: list, query_time: str, topk: int):
        retrieved_results = ____
        answer = ____
        return {"retrieved_results": retrieved_results, "answer": answer}

**정답**

1. `BasicMCPClient(external_mcp_server)`

   해설: 외부 MCP server와 통신할 client를 생성합니다.

2. `McpToolSpec(client=mcp_client)`

   해설: MCP client를 LlamaIndex tool spec으로 감싸 agent가 사용할 수 있게 합니다.

3. `await self.mcp_tool_spec.to_tool_list_async()`

   해설: MCP server가 제공하는 tool 목록을 비동기로 가져옵니다.

4. `Context(self.agent)`

   해설: agent 실행 상태를 유지할 workflow context를 만듭니다.

5. `self.agent.run(question, ctx=self.agent_context)`

   해설: 질문을 agent에 전달해 필요한 tool call을 수행하게 합니다.

6. `await handler`

   해설: agent 실행이 끝난 뒤 최종 응답을 비동기로 받습니다.

7. `await self.mcp_application.query(full_query, verbose=False)`

   해설: query와 query_time을 MCP 기반 KG agent에 전달해 검색 결과를 얻습니다.

8. `await self.retrieve(query, query_time, search_results, topk)`

   해설: inference 단계에서 먼저 MCP retrieval을 수행합니다.

9. `self.reader.generate_response(query, query_time, retrieved_results)`

   해설: MCP 검색 결과를 Reader에 넘겨 최종 답변을 생성합니다.

**전체 해설**

MCP는 외부 도구/API를 agent가 호출할 수 있게 연결하는 방식입니다. RAGwithMCP는 MCP tool call로 KG 결과를 얻고, 그 결과를 Reader가 최종 답변 생성에 사용하도록 연결합니다.